## Prepare questions from NACC for training GRPO

Created by: Sahana Kowshik

Date: 05/06/2025

In [1]:
import pandas as pd
import json
import random
import numpy as np

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"

In [3]:
train = pd.read_csv(f"{directory_path}/training_data/train.csv")
# test = pd.read_csv(f"{directory_path}/training_data/test.csv")

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/2191947696.py:1: DtypeWarning: Columns (20,24,26,28,40,43,45,47,50,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,396,398,400,420,422,431,444,453,492,571,599,607,663,679,696,699,716,727,733,808,819,821,823,825,831,892,947,948,949,957,958,959,960,970,992,995,998,1017,1022,1192,1196,1199,1397,1399,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(f"{directory_path}/training_da

In [4]:
def add_etpr_labels(row):
    value = row['NACCETPR']
    
    # AD
    if value == 1:
        row['ETPR'] = 'AD'

    # LBD
    elif value == 2:
        row['ETPR'] = 'LBD'

    # VD
    elif value == 8:
        row['ETPR'] = 'VD'

    # Prion disease (CJD, other)
    elif value == 12:
        row['ETPR'] = 'PRD'

    # FTLD and its variants, including CBD and PSP, with or without ALS
    elif value in [4, 5, 6, 7]:
        row['ETPR'] = 'FTLD'

    # NPH
    elif value == 14:
        row['ETPR'] = 'NPH'
    
    # SEF: seizure, epilepsy, etc.
    elif value in [17, 18, 23, 26, 27, 28, 29]:
        row['ETPR'] = 'SEF'

    # Psychiatric conditions
    elif value in [19, 20, 21, 22, 24, 25]:
        row['ETPR'] = 'PSY'

    # TBI
    elif value == 13:
        row['ETPR'] = 'TBI'

    # Other degenerative
    elif value not in [88, 99]:
        row['ETPR'] = 'ODE'

    else:
        row['ETPR'] = np.NaN

    return row


train = train.apply(add_etpr_labels, axis=1)

In [5]:
columns = ['NACCID', 'NACCVNUM', 'NACCETPR', 'NACCUDSD', 'NPADNC', 'NPLBOD', 'NPFTDTAU', 'AMYLPET', 'DATSCAN', 'NACCTMCI', 'ETPR', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth']

In [6]:
ids_used = set()

# Train

## Neuropathology

In [7]:
# Neuropathology Question Variants
np_question_variants = [
    "Which of the following pathologies are causing the patient's cognitive impairment based on the clinical information provided?",
    "Using the provided clinical information, determine the pathological causes of the patient's cognitive decline from the options listed below.",
    "Based on the clinical data, identify the neuropathologies responsible for the patient's cognitive impairment by selecting from the given choices.",
    "From the options below, select the disease pathologies causing cognitive impairment as indicated by the clinical information.",
    "Review the provided clinical information and choose the pathologies underlying the patient's cognitive symptoms from the available options."
]

# Answer options for the Neuropathology question
np_options = """A. Alzheimer's disease neuropathology (AD)
B. Lewy body pathology (LBD)
C. Frontotemporal Lobar Degeneration with tau pathology (FTLD-tau) or other tauopathy (FTLD)
D. AD and LBD
E. AD and FTLD
F. LBD and FTLD
G. AD and LBD and FTLD
H. None of the above"""


In [8]:
# Function to add NP ground truth columns
def add_ground_truth(row):
    # Alzheimer's Disease
    if row['NPADNC'] in [2, 3]:
        row['NP_AD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPADNC'] in [0, 1]:
        row['NP_AD'] = 0
        row['NP_AVAIL'] = 1

    # Lewy Body Disease
    if row['NACCLEWY'] in [1, 2, 3, 4]:
        row['NP_LBD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NACCLEWY'] == 0:
        row['NP_LBD'] = 0
        row['NP_AVAIL'] = 1

    # Frontotemporal Degeneration (FTLD-tau or other)
    if row['NPFTDTAU'] == 1:
        row['NP_FTD'] = 1
        row['NP_AVAIL'] = 1
    elif row['NPFTDTAU'] == 0:
        row['NP_FTD'] = 0
        row['NP_AVAIL'] = 1

    return row


In [9]:
# Apply the function across the training set
train = train.apply(add_ground_truth, axis=1)

# Check distribution of available neuropathology labels
train['NP_AVAIL'].value_counts()

NP_AVAIL
1.0    3000
Name: count, dtype: int64

In [10]:
# Keep only patients with neuropathology data available
train_np = train[train['NP_AVAIL'] == 1].reset_index(drop=True)

# Quick preview
train_np.head()

,ABRUPT,ABUSOTHR,ABUSX,ADGCEXOM,ADGCEXR,ADGCGWAS,ADGCRND,AFIBRILL,AFRAID,AGIT,...,WAIS,WEIGHT,WHITEVOL,WMHVOL,WONDRFUL,WRTHLESS,diag_summary,patient_summary,sort_key,sort_year
0,0.0,0.0,NaN,0,88,1,ADC 4,NaN,0.0,NaN,...,95.0,145.0,NaN,NaN,0.0,1.0,"{\n ""Clinician Judgment of Symptoms"": {\n ...","{\n ""Subject Demographics"": {\n ""Liv...",0,NaN
1,NaN,0.0,NaN,0,88,1,ADC 10,0.0,0.0,1.0,...,NaN,179.0,NaN,NaN,0.0,0.0,"{\n ""Clinician Judgment of Symptoms"": {\n ...","{\n ""Subject Demographics"": {\n ""Liv...",0,NaN
2,NaN,NaN,NaN,0,88,1,ADC 10,0.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""Clinician Judgment of Symptoms"": {\n ...","{\n ""Subject Demographics"": {\n ""Liv...",0,NaN
3,NaN,0.0,NaN,0,88,0,88,0.0,0.0,0.0,...,NaN,168.0,NaN,NaN,0.0,0.0,"{\n ""Clinician Judgment of Symptoms"": {\n ...","{\n ""Subject Demographics"": {\n ""Liv...",0,NaN
4,NaN,0.0,NaN,0,88,1,ADC 6,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,"{\n ""Clinician Judgment of Symptoms"": {\n ...","{\n ""Subject Demographics"": {\n ""Liv...",0,NaN


In [11]:
# Function to assign question, options, and ground truth label
def add_np_question(row):
    # Define ground truth answer
    if (row['NP_AD'] == 1) and (row['NP_LBD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "G"
    elif (row['NP_LBD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "F"
    elif (row['NP_AD'] == 1) and (row['NP_FTD'] == 1):
        row['ground_truth'] = "E"
    elif (row['NP_AD'] == 1) and (row['NP_LBD'] == 1):
        row['ground_truth'] = "D"
    elif row['NP_FTD'] == 1:
        row['ground_truth'] = "C"
    elif row['NP_LBD'] == 1:
        row['ground_truth'] = "B"
    elif row['NP_AD'] == 1:
        row['ground_truth'] = "A"
    else:
        row['ground_truth'] = "H"

    # Randomly assign one of the question variants
    row['question'] = random.choice(np_question_variants)
    row['options'] = np_options

    return row

In [12]:
# Apply question and label assignment
train_np = train_np.apply(add_np_question, axis=1)

# Select final columns to keep
train_np = train_np[columns]  # Make sure 'columns' is defined somewhere earlier

# Check ground truth distribution
train_np['ground_truth'].value_counts()

ground_truth
H    1040
A     635
D     552
B     488
C     148
E      57
G      50
F      30
Name: count, dtype: int64

In [13]:
# Update set of NACCIDs used to avoid duplication later
np_ids = set(train_np['NACCID'])

# Optional: Check distribution of diagnosis codes
train_np['NACCUDSD'].value_counts()

NACCUDSD
4    2678
3     322
Name: count, dtype: int64

In [14]:
train_np['Q_TYPE'] = 'Neuropath'

## Amylpet

In [42]:
train[(~train['NACCID'].isin(np_ids)) & (train['AMYLPET'].isin([0,1]))]['AMYLPET'].value_counts()

AMYLPET
1.0    1743
0.0    1643
Name: count, dtype: int64

In [43]:
# Filter training data: exclude patients already used in NP task and keep only those with valid amyloid PET results
train_pet = (
    train
    .loc[
        (~train['NACCID'].isin(np_ids)) & 
        (train['AMYLPET'].isin([0, 1]))
    ]
    .groupby('AMYLPET', group_keys=False)  # Ensure balanced sampling by amyloid status
    .apply(lambda x: x.sample(n=min(len(x), 800), random_state=42))
    .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

# Check the distribution of amyloid PET labels
train_pet['AMYLPET'].value_counts()

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/2467528002.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 800), random_state=42))


AMYLPET
0.0    800
1.0    800
Name: count, dtype: int64

In [44]:
# Question variants for amyloid PET classification
pet_question_variants = [
    "Identify if the patient is amyloid positive based on the provided information.",
    "Determine whether this patient is amyloid positive using the given data.",
    "Analyze the patient information to identify amyloid positivity.",
    "Based on the details provided, identify if the patient shows amyloid positivity.",
    "Using the provided patient data, identify whether the patient is amyloid positive."
]

# Multiple-choice answer options
pet_options = """A. Yes
B. No"""

In [45]:
# Keys related to amyloid biomarkers that should be removed from the patient summary
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [46]:
# Function to add question, options, ground truth, and modify patient summary
def add_pet_question(row):
    # Set ground truth answer
    if row['AMYLPET'] == 1:
        row['ground_truth'] = "A"  # Amyloid positive
    elif row['AMYLPET'] == 0:
        row['ground_truth'] = "B"  # Amyloid negative

    # Randomly assign one of the question variants
    row['question'] = random.choice(pet_question_variants)

    # Remove specific amyloid-related keys from the patient summary
    json_file = json.loads(row['patient_summary'])
    for key in pet_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
            
        if key in json_file.get('CSF evidence', {}):
            json_file['CSF evidence'].pop(key)
    
    # Update patient summary
    row['patient_summary'] = json.dumps(json_file, indent=4)
    
    # Set options
    row["options"] = pet_options

    return row

In [47]:
# Apply the function across the PET training dataset
train_pet = train_pet.apply(add_pet_question, axis=1)

# Select only necessary columns
train_pet = train_pet[columns]

# Quick check: How many times each question variant was picked
print(train_pet['question'].value_counts())

# Check distribution of amyloid PET labels
print(train_pet['AMYLPET'].value_counts())

question
Using the provided patient data, identify whether the patient is amyloid positive.    336
Analyze the patient information to identify amyloid positivity.                       323
Identify if the patient is amyloid positive based on the provided information.        318
Based on the details provided, identify if the patient shows amyloid positivity.      313
Determine whether this patient is amyloid positive using the given data.              310
Name: count, dtype: int64
AMYLPET
0.0    800
1.0    800
Name: count, dtype: int64


In [48]:
print(train_pet['ground_truth'].value_counts())

ground_truth
B    800
A    800
Name: count, dtype: int64


In [49]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
pet_ids = set(train_pet['NACCID'])

# Optional: Check cognitive diagnosis distribution
train_pet['NACCUDSD'].value_counts()

NACCUDSD
4    608
1    575
3    417
Name: count, dtype: int64

In [50]:
train_pet['Q_TYPE'] = 'Amyloid PET'

In [51]:
# print(train[train['NACCID'] == 'NACC867075'].iloc[0]['patient_summary'])

In [53]:
for key in pet_keys_remove:
    for i, row in train_pet.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

## Datscan

In [24]:
# train_dat = train[(~train['NACCID'].isin(np_ids.union(pet_ids))) & (train['DATSCAN'].isin([0,1]))].apply(lambda x: x.sample(n=200, random_state=42)).reset_index(drop=True)

In [54]:
train[(~train['NACCID'].isin(np_ids.union(pet_ids))) & (train['DATSCAN'].isin([0,1]))]['DATSCAN'].value_counts()

DATSCAN
0.0    212
1.0    148
Name: count, dtype: int64

In [55]:
# Filter training data: exclude subjects already used for NP and PET tasks,
# and keep only those with valid DATScan results
train_dat = (
    train
    .loc[
        (~train['NACCID'].isin(np_ids.union(pet_ids))) &
        (train['DATSCAN'].isin([0, 1]))
    ]
    .groupby('DATSCAN', group_keys=False)  # Ensure balanced sampling by DATScan status
    .apply(lambda x: x.sample(n=min(len(x), 70), random_state=42))
    .sample(frac=1, random_state=42)  # Shuffle all rows
    .reset_index(drop=True)
)

# Check the distribution of DATScan labels
train_dat['DATSCAN'].value_counts()


/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/2213973127.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 70), random_state=42))


DATSCAN
1.0    70
0.0    70
Name: count, dtype: int64

In [56]:
# Question variants for DATScan classification
dat_question_variants = [
    "Classify if the patient is DatScan positive based on the provided information.",
    "Using the given data, determine whether the patient is DatScan positive.",
    "Analyze the provided information to classify the patient's DatScan status.",
    "Based on the information given, classify whether the patient is DatScan positive.",
    "From the provided patient details, identify if the patient is DatScan positive."
]

# Multiple-choice answer options
dat_options = """A. Yes
B. No"""

In [57]:
# Keys related to DATScan that should be removed from the patient summary
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]


In [58]:
# Function to add question, options, ground truth, and modify patient summary for DAT task
def add_dat_question(row):
    # Set ground truth answer
    if row['DATSCAN'] == 1:
        row['ground_truth'] = "A"  # DATScan positive
    elif row['DATSCAN'] == 0:
        row['ground_truth'] = "B"  # DATScan negative

    # Randomly assign one of the question variants
    row['question'] = random.choice(dat_question_variants)

    # Remove specific DATScan-related keys from the patient summary
    json_file = json.loads(row['patient_summary'])
    for key in dat_keys_remove:
        if key in json_file.get('Imaging evidence', {}):
            json_file['Imaging evidence'].pop(key)
    
    # Update patient summary
    row['patient_summary'] = json.dumps(json_file, indent=4)
    
    # Set options
    row["options"] = dat_options

    return row


In [59]:
# Apply the function across the DAT training dataset
train_dat = train_dat.apply(add_dat_question, axis=1)

# Select only the necessary columns
train_dat = train_dat[columns]

# Check how many times each question variant was selected
print(train_dat['question'].value_counts())

# Check distribution of DATScan labels
print(train_dat['NACCUDSD'].value_counts())

question
Analyze the provided information to classify the patient's DatScan status.           37
From the provided patient details, identify if the patient is DatScan positive.      28
Based on the information given, classify whether the patient is DatScan positive.    27
Classify if the patient is DatScan positive based on the provided information.       25
Using the given data, determine whether the patient is DatScan positive.             23
Name: count, dtype: int64
NACCUDSD
4    80
3    40
1    20
Name: count, dtype: int64


In [60]:
print(train_dat['ground_truth'].value_counts())

ground_truth
A    70
B    70
Name: count, dtype: int64


In [61]:
# Update set of NACCIDs used to avoid reusing subjects in other tasks
dat_ids = set(train_dat['NACCID'])

In [62]:
train_dat['Q_TYPE'] = 'DATSCAN'

In [63]:
for key in dat_keys_remove:
    for i, row in train_dat.iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

## NACCTMCI

In [64]:
train[~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids))]['NACCTMCI'].value_counts()

NACCTMCI
8    31771
2     4298
1     2297
3     1189
4      746
Name: count, dtype: int64

In [65]:
# Sample MCI patients, excluding those already selected for NP, PET, or DAT tasks
train_mci_sample = (
    train
    .loc[
        ~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids))
    ]
    .groupby('NACCTMCI', group_keys=False)
    .apply(lambda x: x.sample(n=min(max(len(x) - 278, 0), 1000), random_state=42))  # Safe sampling
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check the number of sampled patients per MCI subtype
train_mci_sample['NACCTMCI'].value_counts()

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/241169454.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(max(len(x) - 278, 0), 1000), random_state=42))  # Safe sampling


NACCTMCI
1    1000
8    1000
2    1000
3     911
4     468
Name: count, dtype: int64

In [66]:
train_mci_sample['NACCUDSD'].value_counts()

NACCUDSD
3    3379
1     537
4     463
Name: count, dtype: int64

In [67]:
# Get remaining patients not used yet
remaining_patients = train[
    (~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids))) &
    (~train['NACCID'].isin(train_mci_sample['NACCID'])) & 
    (train['NACCUDSD'].isin([1, 4]))
].reset_index(drop=True)

# Sample up to 2500 from each group (normal controls and dementia)
sampled_nc_de = (
    remaining_patients
    .groupby('NACCUDSD', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 2500), random_state=42))
    .sample(frac=1, random_state=42)  # Shuffle again
    .reset_index(drop=True)
)

# Check distribution
sampled_nc_de['NACCUDSD'].value_counts()

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/2886798048.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 2500), random_state=42))


NACCUDSD
1    2500
4    2500
Name: count, dtype: int64

In [68]:
# Combine MCI and NC/DE samples into one training set
train_mci = (
    pd.concat([train_mci_sample, sampled_nc_de], axis=0)
    .sample(frac=1, random_state=42)  # Shuffle combined dataset
    .reset_index(drop=True)
)

# Quick check
train_mci['NACCTMCI'].value_counts()

NACCTMCI
8    6000
2    1000
1    1000
3     911
4     468
Name: count, dtype: int64

In [69]:
# Question variants for MCI subtype classification
mci_question_variants = [
    "Determine the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.",
    "Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype.",
    "Using the information given, classify the patient's mild cognitive impairment (MCI) subtype.",
    "From the patient details provided, determine which mild cognitive impairment (MCI) subtype applies.",
    "Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype."
]

# Answer options for MCI subtype
mci_options = """A: Amnestic MCI- single domain
B: Amnestic MCI- multiple domain
C: Non-amnestic MCI- single domain
D: Non-amnestic MCI- multiple domain
E: Not applicable (no diagnosis of MCI)"""

In [70]:
# Function to assign ground truth label and MCI question
def add_mci_question(row):
    if row['NACCTMCI'] == 1:
        row['ground_truth'] = "A"
    elif row['NACCTMCI'] == 2:
        row['ground_truth'] = "B"
    elif row['NACCTMCI'] == 3:
        row['ground_truth'] = "C"
    elif row['NACCTMCI'] == 4:
        row['ground_truth'] = "D"
    elif row['NACCTMCI'] == 8:
        row['ground_truth'] = "E"

    # Randomly assign one of the question variants
    row['question'] = random.choice(mci_question_variants)

    row['options'] = mci_options
    return row

In [71]:
# Apply the function to the MCI training set
train_mci = train_mci.apply(add_mci_question, axis=1)

# Select only the columns needed for modeling
train_mci = train_mci[columns]

In [72]:
# Quick checks
print(train_mci['ground_truth'].value_counts())
print(train_mci['question'].value_counts())
print(train_mci['NACCUDSD'].value_counts())

ground_truth
E    6000
B    1000
A    1000
C     911
D     468
Name: count, dtype: int64
question
Analyze the patient's information to identify the correct mild cognitive impairment (MCI) subtype.                      1902
Determine the correct subtype of mild cognitive impairment (MCI) for this patient based on the provided information.    1901
Using the information given, classify the patient's mild cognitive impairment (MCI) subtype.                            1875
Based on the patient's data, identify the appropriate mild cognitive impairment (MCI) subtype.                          1869
From the patient details provided, determine which mild cognitive impairment (MCI) subtype applies.                     1832
Name: count, dtype: int64
NACCUDSD
3    3379
1    3037
4    2963
Name: count, dtype: int64


In [73]:
# Update set of used IDs to avoid reusing these patients elsewhere
mci_ids = set(train_mci['NACCID'])

In [74]:
train_mci['Q_TYPE'] = 'MCI subtype'

## NACCETPR

In [75]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
train_etpr_sample = (
    train
    .loc[
        (~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids).union(mci_ids))) &
        (train['ETPR'] != 'ODE')  # Exclude 'other degenerative etiologies'
    ]
    .groupby('ETPR', group_keys=False)
    .apply(lambda x: x.sample(n=min(max(len(x) - 150, 0), 1000), random_state=42))  # Sample up to 1000
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
train_etpr_sample['ETPR'].value_counts()

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/1123153379.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(max(len(x) - 150, 0), 1000), random_state=42))  # Sample up to 1000


ETPR
FTLD    1000
AD      1000
LBD      763
VD       581
SEF      332
PSY      135
Name: count, dtype: int64

In [76]:
# Find patients not used in train_etpr_sample
remaining_patients_etpr = train[
    (~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids).union(mci_ids))) &
    (~train['NACCID'].isin(train_etpr_sample['NACCID'])) & 
    (train['NACCETPR'].isin([1, 88]))
].reset_index(drop=True)

# Sample up to 5000 from each group
sampled_ad_nc = (
    remaining_patients_etpr
    .groupby('NACCETPR', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 5000), random_state=42))
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Check distribution
sampled_ad_nc['NACCETPR'].value_counts()

/scratch/4953605.1.iris-gpu-pub/ipykernel_319041/824532497.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 5000), random_state=42))


NACCETPR
88    5000
1     5000
Name: count, dtype: int64

In [77]:
# Combine main etiology samples with sampled AD/NC cases
train_etpr = (
    pd.concat([train_etpr_sample, sampled_ad_nc], axis=0)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Quick check
print(train_etpr['ETPR'].value_counts())
print(train_etpr['NACCUDSD'].value_counts())

ETPR
AD      6000
FTLD    1000
LBD      763
VD       581
SEF      332
PSY      135
Name: count, dtype: int64
NACCUDSD
4    6357
1    5000
3    2454
Name: count, dtype: int64


In [78]:
# Question variants for primary etiology identification
etiology_question_variants = [
    "Identify the primary etiology of the patient's cognitive impairment using the information provided.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment.",
    "From the given patient information, select the primary underlying etiology for the cognitive symptoms.",
    "Using the data provided, identify the dominant cause of the patient's cognitive decline.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information."
]

# Answer options for primary etiology prediction
etiology_options = """A: Alzheimer's disease (AD)
B: Lewy body disease (LBD)
C: Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTD)
D: Vascular brain injury or vascular dementia including stroke (VD)
E: Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)
F: Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
G: Not applicable (no cognitive impairment)"""

In [79]:
# Function to assign ground truth label and MCI question
def add_etpr_question(row):
    if row['ETPR'] == 'AD':
        row['ground_truth'] = "A"
    elif row['ETPR'] == 'LBD':
        row['ground_truth'] = "B"
    elif row['ETPR'] == 'FTLD':
        row['ground_truth'] = "C"
    elif row['ETPR'] == 'VD':
        row['ground_truth'] = "D"
    elif row['ETPR'] == 'SEF':
        row['ground_truth'] = "E"
    elif row['ETPR'] == 'PSY':
        row['ground_truth'] = "F"
    else:
        row['ground_truth'] = "G"

    # Randomly assign one of the question variants
    row['question'] = random.choice(etiology_question_variants)

    row['options'] = etiology_options
    return row

In [80]:
# Apply the function to the MCI training set
train_etpr = train_etpr.apply(add_etpr_question, axis=1)

# Select only the columns needed for modeling
train_etpr = train_etpr[columns]

In [81]:
print(train_etpr['ground_truth'].value_counts())
print(train_etpr['question'].value_counts())
print(train_etpr['NACCUDSD'].value_counts())

ground_truth
A    6000
G    5000
C    1000
B     763
D     581
E     332
F     135
Name: count, dtype: int64
question
Identify the primary etiology of the patient's cognitive impairment using the information provided.                2882
From the given patient information, select the primary underlying etiology for the cognitive symptoms.             2755
Determine the primary etiology underlying the patient's cognitive impairment based on the provided information.    2747
Based on the clinical details, determine the main cause of the patient's cognitive impairment.                     2721
Using the data provided, identify the dominant cause of the patient's cognitive decline.                           2706
Name: count, dtype: int64
NACCUDSD
4    6357
1    5000
3    2454
Name: count, dtype: int64


In [82]:
# Update set of used IDs to avoid reusing these patients elsewhere
etpr_ids = set(train_etpr['NACCID'])

In [83]:
train_etpr['Q_TYPE'] = 'Primary etiology'

## NACCUDSD

In [84]:
train[~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids).union(mci_ids).union(etpr_ids))]['NACCUDSD'].value_counts()

NACCUDSD
1    9432
4    4982
3    2697
Name: count, dtype: int64

In [85]:
# Filter and sample patients for etiology prediction task,
# excluding those already selected for NP, PET, DAT, or MCI tasks
train_cog = (
    train
    .loc[
        (~train['NACCID'].isin(np_ids.union(pet_ids).union(dat_ids).union(mci_ids).union(etpr_ids)))
    ]
    .sample(frac=1, random_state=42)  # Shuffle the sampled rows
    .reset_index(drop=True)
)

# Check sampled distribution
train_cog['NACCUDSD'].value_counts()

NACCUDSD
1    9432
4    4982
3    2697
Name: count, dtype: int64

In [86]:
# Question variants for primary etiology identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Answer options for primary etiology prediction
cog_options = """A: Normal Cognition (NC)
B: Mild Cognitive Impairment (MCI)
C: Dementia (DE)"""

In [87]:
# Function to assign ground truth label and MCI question
def add_cog_question(row):
    if row['NACCUDSD'] == 1:
        row['ground_truth'] = "A"
    if row['NACCUDSD'] == 3:
        row['ground_truth'] = "B"
    if row['NACCUDSD'] == 4:
        row['ground_truth'] = "C"
        
    # Randomly assign one of the question variants
    row['question'] = random.choice(cog_question_variants)

    row['options'] = cog_options
    return row

In [88]:
# Apply the function to the MCI training set
train_cog = train_cog.apply(add_cog_question, axis=1)

# Select only the columns needed for modeling
train_cog = train_cog[columns]

In [89]:
print(train_cog['ground_truth'].value_counts())
print(train_cog['question'].value_counts())

ground_truth
A    9432
C    4982
B    2697
Name: count, dtype: int64
question
Based on the provided clinical details, classify the patient's cognitive status.           3488
Analyze the patient's information and identify their cognitive status.                     3454
Identify the patient's cognitive status based on the given information.                    3434
Using the information provided, determine the patient's cognitive status.                  3418
From the information available, determine the correct cognitive status for the patient.    3317
Name: count, dtype: int64


In [90]:
# Update set of used IDs to avoid reusing these patients elsewhere
cog_ids = set(train_cog['NACCID'])

In [91]:
train_cog['Q_TYPE'] = 'Cognitive status'

## Combine

In [92]:
train_combined = (
    pd.concat([train_np, train_pet, train_dat, train_mci, train_etpr, train_cog], axis=0)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [93]:
len(train_combined)

45041

In [94]:
question_df = train_combined.groupby(['question', 'Q_TYPE']).size().reset_index(name='count').sort_values(by=['Q_TYPE', 'count'])

In [95]:
# Save as CSV
csv_path = f"{directory_path}/training_data/training_data_grpo/train_question_counts.csv"
question_df.to_csv(csv_path, index=False)

In [96]:
train_combined["ID"] = train_combined["NACCID"]
train_combined.drop(['NACCID'], axis=1, inplace=True)

In [97]:
req_columns = ['ID', 'patient_summary', 'diag_summary', 'question', 'options', 'ground_truth', 'Q_TYPE']
train_combined = train_combined[req_columns]

In [98]:
train_combined.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,Q_TYPE
0,NACC167653,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...",Identify if the patient is amyloid positive ba...,A. Yes\nB. No,B,Amyloid PET
1,NACC965610,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the patient's data, identify the appr...",A: Amnestic MCI- single domain\nB: Amnestic MC...,E,MCI subtype
2,NACC133624,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the given patient information, select the...",A: Alzheimer's disease (AD)\nB: Lewy body dise...,A,Primary etiology
3,NACC844302,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the given patient information, select the...",A: Alzheimer's disease (AD)\nB: Lewy body dise...,A,Primary etiology
4,NACC884014,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the information available, determine the ...",A: Normal Cognition (NC)\nB: Mild Cognitive Im...,A,Cognitive status


In [99]:
def remove_cop_data(text):
    patient_file_json = json.loads(text)
    if 'Co-participant Demographics' in patient_file_json:
        del patient_file_json['Co-participant Demographics']
        text = json.dumps(patient_file_json, indent=4)
        
    return text

train_combined["patient_summary"] = train_combined["patient_summary"].apply(remove_cop_data)

In [100]:
train_combined.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,Q_TYPE
0,NACC167653,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...",Identify if the patient is amyloid positive ba...,A. Yes\nB. No,B,Amyloid PET
1,NACC965610,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","Based on the patient's data, identify the appr...",A: Amnestic MCI- single domain\nB: Amnestic MC...,E,MCI subtype
2,NACC133624,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the given patient information, select the...",A: Alzheimer's disease (AD)\nB: Lewy body dise...,A,Primary etiology
3,NACC844302,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the given patient information, select the...",A: Alzheimer's disease (AD)\nB: Lewy body dise...,A,Primary etiology
4,NACC884014,"{\n ""Subject Demographics"": {\n ""Liv...","{\n ""Clinician Judgment of Symptoms"": {\n ...","From the information available, determine the ...",A: Normal Cognition (NC)\nB: Mild Cognitive Im...,A,Cognitive status


In [102]:
# print(train_combined.iloc[0]['patient_summary'])

In [10]:
train_combined.to_csv(f"{directory_path}/training_data/training_data_grpo/train.csv", index=False)